# 02. Carga: dengue no Postgres

Leva para o banco as notificações baixadas no `01_coleta_dengue.ipynb` (opção B, CSV anual).

| Entrada | Saída no banco `dengue-eda` |
|---|---|
| `data/raw/dengue_csv/ano=AAAA/DENGBR{aa}.csv` | `sinan_dengue`: uma linha por notificação |
| | `sinan_dengue_carga`: registro de cada ano carregado |

**Como fica a tabela `sinan_dengue`**
- É o dado bruto: Brasil inteiro, sem filtro e sem limpeza.
- Tem todas as colunas que aparecem nos CSVs (196), como texto, mais `ano_arquivo` (o ano do arquivo de origem).
  Uma coluna que não existe num período fica vazia nele.
- É dividida por ano, então recarregar um ano não mexe nos outros.

**Como fica a tabela `sinan_dengue_carga`**
- Uma linha por ano carregado: arquivo de origem, número de linhas e data da carga.
- Serve para não carregar duas vezes: rodando de novo, os anos já carregados são pulados.

**Neste notebook**
1. Conexão
2. Carga
3. Conferência

In [ ]:
import sys
from pathlib import Path
RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "config.py").exists())
sys.path.append(str(RAIZ / "src"))

import pandas as pd
import psycopg
from carga.dengue import carga_dengue
from coleta.dengue import coleta_dengue_csv as dengue_csv
from config import ANO_INICIO, ANO_FIM_COLETA, BANCO

def consulta(q):
    with psycopg.connect(**BANCO) as con:
        cur = con.execute(q)
        return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])

## 1. Conexão
Banco Postgres local. Os dados de acesso ficam no arquivo `.env` na raiz do projeto (modelo em `.env.example`).


In [ ]:
consulta("select version()")

## 2. Carga
Carrega um ano por vez (leva poucos minutos no total). Para carregar de novo os anos já carregados, use `REFAZER = True`.

In [ ]:
REFAZER = False
carga_dengue.carregar(range(ANO_INICIO, ANO_FIM_COLETA + 1), refazer=REFAZER)

## 3. Conferência

### 3.1 O banco tem tudo o que estava no CSV?
Número de linhas por ano no banco e no CSV. Devem ser iguais.

In [ ]:
no_banco = consulta("select ano_arquivo as ano, count(*) as banco from sinan_dengue group by 1 order by 1").set_index("ano")
no_csv = pd.Series({ano: len(pd.read_csv(dengue_csv.DESTINO / f"ano={ano}" / f"DENGBR{ano % 100:02d}.csv",
                                         usecols=["NU_ANO"], dtype=str))
                    for ano in range(ANO_INICIO, ANO_FIM_COLETA + 1)}, name="csv")
linhas = no_banco.join(no_csv)
linhas["igual"] = linhas["banco"] == linhas["csv"]
print(f"Total no banco: {linhas['banco'].sum():,} | anos iguais ao CSV: {linhas['igual'].sum()} de {len(linhas)}")
linhas

### 3.2 Campos principais
% de linhas preenchidas em alguns campos importantes, por ano. Idade e classificação mudam de nome em 2007:
`nu_idade` e `con_classi` até 2006; `nu_idade_n` e `classi_fin` depois.

In [ ]:
consulta("""
    select ano_arquivo as ano, count(*) as linhas,
           round(100.0 * count(id_mn_resi) / count(*)) as id_mn_resi,
           round(100.0 * count(dt_sin_pri) / count(*)) as dt_sin_pri,
           round(100.0 * count(cs_sexo)    / count(*)) as cs_sexo,
           round(100.0 * count(nu_idade)   / count(*)) as nu_idade,
           round(100.0 * count(nu_idade_n) / count(*)) as nu_idade_n,
           round(100.0 * count(con_classi) / count(*)) as con_classi,
           round(100.0 * count(classi_fin) / count(*)) as classi_fin
    from sinan_dengue group by 1 order by 1""").set_index("ano")

### 3.3 Espaço ocupado no banco

In [ ]:
consulta("""
    select pg_size_pretty(sum(pg_total_relation_size(inhrelid))) as tamanho_total,
           count(*) as particoes
    from pg_inherits where inhparent = 'sinan_dengue'::regclass""")

### 3.4 Exemplo: notificações de quem mora no Nordeste
Municípios de residência com código começando de 21 a 29 (o recorte do Nordeste é feito na limpeza).

In [ ]:
consulta("""
    select ano_arquivo as ano, count(*) as notificacoes_ne
    from sinan_dengue where left(id_mn_resi, 2) between '21' and '29'
    group by 1 order by 1""").set_index("ano").T